<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/08_text_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent Google Drive storage
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Configure Git identity and remote access
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "preethamgowda837@gmail.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Clone if missing, or pull latest changes
%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

# 4. Enforce Python module search path resolution
src_path_str = str(SRC_DIR.resolve())
if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

# 5. Install required external dependencies
!pip install -q transformers pyarrow

print("=" * 60)
print(f"Working Directory: {os.getcwd()}")
print(f"Module Path:       {src_path_str}")
print("Environment configuration complete.")
print("=" * 60)

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 108 (delta 40), reused 67 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 497.41 KiB | 2.54 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/DP-MMFL
Working Directory: /content/DP-MMFL
Module Path:       /content/DP-MMFL/src
Environment configuration complete.


In [ ]:
from pathlib import Path
import pandas as pd
import torch

# Load the verified manifest
PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"

manifest = pd.read_parquet(MANIFEST_PATH)
print("Manifest loaded successfully.")
print(f"Total records: {manifest.shape[0]:,}")
print(f"Columns:       {manifest.shape[1]}")

# Import from the newly verified package path
from dp_mmfl.data.text import ClinicalTextTokenizer, MODEL_NAME, MAX_LENGTH

tokenizer_module = ClinicalTextTokenizer(model_name=MODEL_NAME, max_length=MAX_LENGTH)
print(f"Tokenizer initialized: {MODEL_NAME} | Fixed length: {MAX_LENGTH}")

Manifest loaded successfully.
Total records: 223,462
Columns:       58


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer initialized: emilyalsentzer/Bio_ClinicalBERT | Fixed length: 384


In [ ]:
sample_reports = manifest["report_clean"].head(4).tolist()
encoded = tokenizer_module.encode_batch(sample_reports)

input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

print("--- Tensor Metrics & Data Types ---")
print("input_ids shape:      ", input_ids.shape)
print("attention_mask shape: ", attention_mask.shape)
print("input_ids dtype:      ", input_ids.dtype)
print("attention_mask dtype: ", attention_mask.dtype)

# Structural assertions
assert input_ids.shape == torch.Size([4, 384]), f"Expected [4, 384], got {input_ids.shape}"
assert attention_mask.shape == torch.Size([4, 384]), f"Expected [4, 384], got {attention_mask.shape}"
assert input_ids.dtype == torch.int64, f"Expected torch.int64, got {input_ids.dtype}"
assert attention_mask.dtype == torch.int64, f"Expected torch.int64, got {attention_mask.dtype}"

# Padding check
non_padding_counts = attention_mask.sum(dim=1)
print("\n--- Active vs Padding Tokens Per Sample ---")
for idx, count in enumerate(non_padding_counts.tolist()):
    print(f"Sample {idx + 1}: {count:3d} active tokens | {384 - count:3d} padding tokens")

assert (non_padding_counts <= 384).all(), "Sequence length exceeded 384 boundary!"
assert (non_padding_counts > 0).all(), "Zero active tokens detected in batch!"
print("\nTokenization pipeline verification complete: All checks passed.")

--- Tensor Metrics & Data Types ---
input_ids shape:       torch.Size([4, 384])
attention_mask shape:  torch.Size([4, 384])
input_ids dtype:       torch.int64
attention_mask dtype:  torch.int64

--- Active vs Padding Tokens Per Sample ---
Sample 1: 192 active tokens | 192 padding tokens
Sample 2: 180 active tokens | 204 padding tokens
Sample 3: 180 active tokens | 204 padding tokens
Sample 4: 157 active tokens | 227 padding tokens

Tokenization pipeline verification complete: All checks passed.


In [ ]:
print(manifest.columns.tolist())

['sample_id', 'path_to_image', 'path_to_dcm', 'deid_patient_id', 'report', 'age', 'sex', 'race', 'ethnicity', 'split', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'No Finding', 'target_Enlarged Cardiomediastinum', 'target_Cardiomegaly', 'target_Lung Opacity', 'target_Lung Lesion', 'target_Edema', 'target_Consolidation', 'target_Pneumonia', 'target_Atelectasis', 'target_Pneumothorax', 'target_Pleural Effusion', 'target_Pleural Other', 'target_Fracture', 'target_Support Devices', 'mask_Enlarged Cardiomediastinum', 'mask_Cardiomegaly', 'mask_Lung Opacity', 'mask_Lung Lesion', 'mask_Edema', 'mask_Consolidation', 'mask_Pneumonia', 'mask_Atelectasis', 'mask_Pneumothorax', 'mask_Pleural Effusion', 'mask_Pleural Other', 'mask_Fracture', 'mask_Support Devices', 'experiment_split', 'frontal_lateral', 'ap_pa', 'patient_report_da

In [ ]:
target_columns = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

for col in target_columns:
    print(
        col,
        "-> target:", f"{col}_target" in manifest.columns,
        "| mask:", f"{col}_mask" in manifest.columns,
    )

Enlarged Cardiomediastinum -> target: False | mask: False
Cardiomegaly -> target: False | mask: False
Lung Opacity -> target: False | mask: False
Lung Lesion -> target: False | mask: False
Edema -> target: False | mask: False
Consolidation -> target: False | mask: False
Pneumonia -> target: False | mask: False
Atelectasis -> target: False | mask: False
Pneumothorax -> target: False | mask: False
Pleural Effusion -> target: False | mask: False
Pleural Other -> target: False | mask: False
Fracture -> target: False | mask: False
Support Devices -> target: False | mask: False
